<a href="https://colab.research.google.com/github/lalaayyey/BigData26_B_2411533019_AdilaBintang/blob/main/Copy_of_BD_B_P02_2411533019_AdilaBintang_ipynb_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
# Install library faker (J. Persiapan Praktikum)
!pip install faker -q

In [51]:
# K-1. Import Library dan Inisialisasi
import numpy as np
import pandas as pd
from faker import Faker
import random

In [52]:
# K-2. Membuat Dataset Sintetis (Simulasi Data Acquisition)
# ==========================================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar}",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)
    metode = random.choice(metode_bayar)

    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper()

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None]) # rating opsional

    rows.append({
        "transaction_id": trx_id,
        "customer_name": nama_pelanggan,
        "product_name": produk.strip(),
        "category": kategori,
        "price": harga,
        "quantity": qty,
        "payment_method": metode,
        "transaction_date": tanggal,
        "shipping_city": kota,
        "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


In [53]:
# K-3. Deteksi dan Penanganan Missing Value

print(df.isnull().sum())

df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")
print("Jumlah baris setelah drop_duplicates():", len(df))

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64
Jumlah baris setelah drop_duplicates(): 495


In [54]:
# K-4. Deteksi dan Penanganan Duplicate

print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


In [55]:
# K-5. Koreksi Tipe Data dan Standardisasi Format

# a. Standardisasi teks kategorikal
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

# b. Koreksi tipe data pada kolom price
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

# c. Standardisasi format tanggal ke YYYY-MM-DD
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

# d. Finalisasi tipe data
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

In [56]:
# K-6. Ekspor Dataset Bersih

df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


In [57]:
# LATIHAN 1

# Jalankan ulang pipeline generator dengan SEED = 7
SEED_7 = 7
np.random.seed(SEED_7)
random.seed(SEED_7)
fake = Faker("id_ID")  # Menggunakan id_ID (pakai underscore)
Faker.seed(SEED_7)

rows_7 = []
for i in range(1, 501):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"])
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    harga_variants = [str(harga_dasar), f"Rp{harga_dasar:,}".replace(",", "."), f"{harga_dasar}.0", f" {harga_dasar}"]
    harga = random.choice(harga_variants)

    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)
    metode = random.choice(["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"])

    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])

    rows_7.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df_7 = pd.DataFrame(rows_7)

for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df_7.sample(frac=frac, random_state=SEED_7).index
    df_7.loc[idx, col] = np.nan

dup_rows_7 = df_7.sample(n=15, random_state=SEED_7)
df_7 = pd.concat([df_7, dup_rows_7], ignore_index=True)
df_7 = df_7.sample(frac=1, random_state=SEED_7).reset_index(drop=True)

mentah_7_len = len(df_7)

# Cleaning untuk SEED 7
df_7 = df_7.drop_duplicates()
df_7 = df_7.dropna(subset=["customer_name", "payment_method"])
bersih_7_len = len(df_7)

print("Latihan 1 Output:")
print("SEED=42 -> Mentah: 515 baris, Bersih: 490 baris")
print(f"SEED=7  -> Mentah: {mentah_7_len} baris, Bersih: {bersih_7_len} baris\n")


Latihan 1 Output:
SEED=42 -> Mentah: 515 baris, Bersih: 490 baris
SEED=7  -> Mentah: 515 baris, Bersih: 490 baris



In [58]:
df.head()

,transaction_id,customer_name,product_name,category,price,quantity,payment_method,transaction_date,shipping_city,rating
0,TRX00305,Ophelia Hartati,Quo Basic,Buku,50000.0,4,Kartu Kredit,2026-07-15,Blitar,1.0
1,TRX00500,Cut Maya Wijayanti,Delectus,Elektronik,500000.0,1,E-Wallet,2026-07-11,Lubuklinggau,NaN
2,TRX00442,"Dr. Ridwan Utama, M.Pd",Animi Max,Elektronik,25000.0,1,COD,2026-08-27,Probolinggo,2.0
3,TRX00154,"Viman Suwarno, S.H.",Consectetur,Fashion,250000.0,1,COD,2026-07-29,Tual,4.0
5,TRX00132,Ir. Ilyas Setiawan,Ducimus Pro,Rumah Tangga,250000.0,4,COD,2026-09-17,Subulussalam,2.0


In [59]:
# LATIHAN 2

df["is_valid_price"] = df["price"] > 0
invalid_prices = df[df["is_valid_price"] == False]

print("Latihan 2 Output:")
print("Jumlah transaksi dengan harga tidak valid (price <= 0):", len(invalid_prices))
print("Status keabsahan harga:", "Semua harga valid" if len(invalid_prices) == 0 else "Ada harga tidak valid")
print()

Latihan 2 Output:
Jumlah transaksi dengan harga tidak valid (price <= 0): 0
Status keabsahan harga: Semua harga valid



In [60]:
df.head()

,transaction_id,customer_name,product_name,category,price,quantity,payment_method,transaction_date,shipping_city,rating,is_valid_price
0,TRX00305,Ophelia Hartati,Quo Basic,Buku,50000.0,4,Kartu Kredit,2026-07-15,Blitar,1.0,True
1,TRX00500,Cut Maya Wijayanti,Delectus,Elektronik,500000.0,1,E-Wallet,2026-07-11,Lubuklinggau,NaN,True
2,TRX00442,"Dr. Ridwan Utama, M.Pd",Animi Max,Elektronik,25000.0,1,COD,2026-08-27,Probolinggo,2.0,True
3,TRX00154,"Viman Suwarno, S.H.",Consectetur,Fashion,250000.0,1,COD,2026-07-29,Tual,4.0,True
5,TRX00132,Ir. Ilyas Setiawan,Ducimus Pro,Rumah Tangga,250000.0,4,COD,2026-09-17,Subulussalam,2.0,True


In [61]:
# LATIHAN 3

kategori_counts = df["category"].value_counts()

print("Latihan 3 Output:")
print("Jumlah transaksi per kategori:")
print(kategori_counts)

Latihan 3 Output:
Jumlah transaksi per kategori:
category
Olahraga        97
Kesehatan       91
Elektronik      89
Buku            82
Fashion         66
Rumah Tangga    65
Name: count, dtype: Int64


In [62]:
df.head()

,transaction_id,customer_name,product_name,category,price,quantity,payment_method,transaction_date,shipping_city,rating,is_valid_price
0,TRX00305,Ophelia Hartati,Quo Basic,Buku,50000.0,4,Kartu Kredit,2026-07-15,Blitar,1.0,True
1,TRX00500,Cut Maya Wijayanti,Delectus,Elektronik,500000.0,1,E-Wallet,2026-07-11,Lubuklinggau,NaN,True
2,TRX00442,"Dr. Ridwan Utama, M.Pd",Animi Max,Elektronik,25000.0,1,COD,2026-08-27,Probolinggo,2.0,True
3,TRX00154,"Viman Suwarno, S.H.",Consectetur,Fashion,250000.0,1,COD,2026-07-29,Tual,4.0,True
5,TRX00132,Ir. Ilyas Setiawan,Ducimus Pro,Rumah Tangga,250000.0,4,COD,2026-09-17,Subulussalam,2.0,True
